# Person A — Notebook 3 v2: KL-Stabilized Reconsolidation (Polished)
**CS 590NN | Amogh | Apr 18 — polished rerun**

Polished version of Notebook 3 addressing issues found in the Apr 18 run.

**Fixes vs v1:**
1. **kl_final at 1-step was always 0.0** — measurement bug, not training bug. Fixed by moving KL measurement to after the loop with a fresh forward pass.
2. **Per-sample KL trajectory tracking** (`kl_history`) — see KL rise during training.
3. **% samples flipped metric** — cleaner success indicator than mean P(target_new).
4. **Gradient clipping at max_norm=1.0** — prevents runaway at higher step counts.
5. **Per-sample try/except** — one failure won't kill the whole run.
6. **Inline figures** — KL vs steps, edit_success vs steps with error bars.
7. **Richer harness output** — per-row `p_new_history` and `kl_history` for downstream analysis.

**Output:** `week5_harness_output_kl_v2.json` — drop-in for Aneesh's harness,
same schema as v1 plus the new history fields.

**Runtime estimate:** ~55 min on A100 for 50 samples × 4 step counts with
per-step KL measurement.


### 3.0 Install
Run once. Runtime restarts automatically.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens", "transformers", "datasets", "accelerate", "einops", "jaxtyping"])
pip(["matplotlib"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)


### 3.1 Imports

In [ ]:
import torch, json, re, requests, random, traceback
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 0), \
    "NumPy >= 2.0 — re-run cell 3.0 and restart."
print(f"NumPy  : {np.__version__}")

torch.manual_seed(42)
random.seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
FIGS_DIR = Path("figures")
FIGS_DIR.mkdir(exist_ok=True)
print("Imports OK")


### 3.2 Load Model
`set_use_attn_result(True)` so attention heads can actually be edited (from
Notebook 1 v2+).

In [ ]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Config : {model.cfg.n_layers} layers | {model.cfg.n_heads} heads | d_model={model.cfg.d_model}")
print(f"Memory : {free/1e9:.1f} GB free after load")


### 3.3 Upload Circuit Log
Upload `week2_circuit_log_v2.1.json` (threshold-fixed ACDC circuits).

In [ ]:
from google.colab import files

print("Upload week2_circuit_log_v2.1.json from Notebook 1 v2.1...")
uploaded = files.upload()
log_path = next(iter(uploaded.keys()))
with open(log_path) as f:
    circuit_log = json.load(f)

print(f"Loaded {len(circuit_log)} circuit entries from {log_path}")
print(f"Sample 0: attn={circuit_log[0]['attn_heads'][:3]}  mlp={circuit_log[0]['mlp_layers'][:5]}")
n_with_attn = sum(1 for e in circuit_log if e['n_attn'] > 0)
n_clamped = sum(1 for e in circuit_log if e.get('effect_clamped', False))
print(f"Circuits with attention heads: {n_with_attn}/{len(circuit_log)}")
print(f"Samples flagged effect_clamped: {n_clamped}/{len(circuit_log)}")
avg_attn = sum(e['n_attn'] for e in circuit_log) / len(circuit_log)
avg_mlp = sum(e['n_mlp']  for e in circuit_log) / len(circuit_log)
print(f"Avg circuit size: n_attn={avg_attn:.1f}, n_mlp={avg_mlp:.1f}")


### 3.4 Load CounterFact (50 samples)

In [ ]:
@dataclass
class EditSample:
    prompt:          str
    target_new:      str
    target_true:     str
    related_prompts: List[str] = field(default_factory=list)

print("Downloading CounterFact from ROME project source...")
raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
print(f"Downloaded {len(raw_data)} records")

N_SAMPLES = 50

def parse_counterfact(raw):
    neighborhood = raw.get("neighborhood_prompts", [])
    related = [p for p in neighborhood[:3] if isinstance(p, str)]
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" "  + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=related,
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples")
print(f"Example: '{cf_samples[0].prompt}' -> true='{cf_samples[0].target_true}'")


### 3.5 Neutral Prompt Anchor Set for KL

Same 32-prompt neutral set as v1. Reference log-probs are cached per sample
with original weights, then KL is computed against those cached logits at
every step. Avoids holding a second model copy.

In [ ]:
NEUTRAL_PROMPTS = [
    "The sum of two and three is", "Twelve divided by four equals",
    "The square root of nine is", "Ten times ten equals",
    "The capital of Japan is", "The largest ocean on Earth is the",
    "Mount Everest is located in the", "The Amazon River flows through",
    "Water boils at one hundred degrees", "The chemical symbol for gold is",
    "Plants produce oxygen through a process called", "The Earth orbits the",
    "A week contains seven", "The primary colors are red, blue, and",
    "Humans have two lungs and one", "A triangle has three",
    "The opposite of hot is", "A baby cat is called a",
    "The past tense of run is", "A group of fish is called a",
    "The Industrial Revolution began in the eighteenth",
    "The Renaissance was a period of cultural",
    "World War Two ended in the year", "The Wright Brothers invented the",
    "A computer's main processor is the", "The world wide web was invented by",
    "An algorithm is a sequence of", "The unit of electrical resistance is the",
    "Roses are typically red while violets are",
    "The sky appears blue because of light",
    "A year contains twelve", "The freezing point of water is zero degrees",
]
assert len(NEUTRAL_PROMPTS) == 32

def cache_reference_logits(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            tokens = model.to_tokens(p)
            logits = model(tokens)
            ref_lp = torch.log_softmax(logits[0, -1, :], dim=-1).detach().clone()
            cache.append((tokens, ref_lp))
    return cache

print(f"Neutral prompt set: {len(NEUTRAL_PROMPTS)} prompts")
print(f"Vocab size: {model.cfg.d_vocab}")


### 3.6 4-Step Reconsolidation (Polished)

Key fixes in `run_edit_with_kl_polished`:

1. **KL measurement moved outside the loop.** At 1 step in v1, `final_kl` was
   written before any Adam step ran, so it recorded pre-edit KL (= 0 since
   refs cached from same weights). Now we do a fresh `kl_loss_neutral` call
   at the end after all steps complete.

2. **kl_history tracks every step's KL.** Measured AFTER each Adam step so
   it reflects post-update drift. This is what we want for the paper figure.

3. **p_new_history tracks edit_success every step.** Lets us plot the edit
   trajectory.

4. **Gradient clipping at max_norm=1.0** — prevents runaway at 20 steps.

5. **Try/except wrapping.** Failed samples don't kill the run.

In [ ]:
def restore_weights(model, state):
    """Restore weights in-place — no temporary GPU allocation."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            param.copy_(state[name])
    torch.cuda.empty_cache()


def get_circuit_params(model, circuit_attn, circuit_mlp):
    params = []
    for (layer, head) in circuit_attn:
        try:
            params.append(model.blocks[layer].attn.W_O)
        except Exception:
            pass
    for layer in circuit_mlp:
        try:
            params.append(model.blocks[layer].mlp.W_in)
            params.append(model.blocks[layer].mlp.W_out)
        except Exception:
            pass
    return params


def contrastive_loss(model, sample):
    tokens  = model.to_tokens(sample.prompt)
    new_id  = model.to_tokens(sample.target_new,  prepend_bos=False)[0, 0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0, 0]
    logits  = model(tokens)
    lp      = torch.nn.functional.log_softmax(logits[0, -1, :], dim=-1)
    loss    = -lp[new_id] + lp[true_id]
    p_new   = lp[new_id].exp().item()
    p_true  = lp[true_id].exp().item()
    del tokens, logits
    return loss, p_new, p_true


def kl_loss_neutral(model, ref_cache):
    """Mean KL(p_ref || p_edit) across the cached neutral prompts."""
    total = 0.0
    for tokens, ref_lp in ref_cache:
        logits = model(tokens)
        edit_lp = torch.log_softmax(logits[0, -1, :], dim=-1)
        kl = (ref_lp.exp() * (ref_lp - edit_lp)).sum()
        total = total + kl
        del logits, edit_lp
    return total / len(ref_cache)


def kl_loss_neutral_eval(model, ref_cache) -> float:
    """Same as kl_loss_neutral but returns a Python float and uses no_grad.
    For measurement only, not training."""
    total = 0.0
    model.eval()
    with torch.no_grad():
        for tokens, ref_lp in ref_cache:
            logits = model(tokens)
            edit_lp = torch.log_softmax(logits[0, -1, :], dim=-1)
            kl = (ref_lp.exp() * (ref_lp - edit_lp)).sum().item()
            total += kl
            del logits, edit_lp
    return total / len(ref_cache)


def run_edit_with_kl_polished(
    model,
    sample,
    circuit_attn: List[Tuple[int, int]],
    circuit_mlp: List[int],
    n_steps: int,
    lr: float = 5e-3,
    beta_kl: float = 0.1,
    grad_clip: float = 1.0,
    ref_cache=None,
) -> Dict:
    """KL-stabilized 4-step reconsolidation with polished measurement.

    v2 fixes:
      - KL measured AFTER the loop with a fresh forward pass (fixes v1's
        kl_final=0.0 bug at n_steps=1).
      - Gradient clipping prevents runaway.
      - Per-step KL and p_new tracking for trajectory plots.
    """
    params = get_circuit_params(model, circuit_attn, circuit_mlp)
    if not params:
        params = [p for layer in model.blocks
                  for p in [layer.mlp.W_in, layer.mlp.W_out]]

    for p in model.parameters():
        p.requires_grad_(False)
    for p in params:
        p.requires_grad_(True)

    optimizer = torch.optim.Adam(params, lr=lr)

    # Baseline (pre-edit)
    model.eval()
    with torch.no_grad():
        _, baseline, _ = contrastive_loss(model, sample)

    # Initial KL should be ~0 since ref_cache was computed from these weights.
    # We record it as a sanity check.
    if ref_cache is not None:
        kl_initial = kl_loss_neutral_eval(model, ref_cache)
    else:
        kl_initial = 0.0

    # Per-step trajectory
    kl_history = []
    p_new_history = []

    for step in range(n_steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        # Contrastive push/pull on trigger prompt
        loss, _, _ = contrastive_loss(model, sample)

        # KL re-freeze on neutral prompts
        if ref_cache is not None and beta_kl > 0:
            kl = kl_loss_neutral(model, ref_cache)
            loss = loss + beta_kl * kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        optimizer.step()
        del loss

        # Measurement AFTER the step — this is the key fix vs v1
        step_kl = kl_loss_neutral_eval(model, ref_cache) if ref_cache is not None else 0.0
        _, step_p_new, _ = contrastive_loss(model, sample)
        kl_history.append(step_kl)
        p_new_history.append(step_p_new)

    torch.cuda.empty_cache()

    # Final measurements (also equal to the last entry in the history, but
    # keeping as a separate named field for backward-compat with v1 schema)
    model.eval()
    with torch.no_grad():
        _, final_p_new, final_p_true = contrastive_loss(model, sample)

    kl_final = kl_history[-1] if kl_history else 0.0

    for p in model.parameters():
        p.requires_grad_(True)

    return {
        "edit_success":    final_p_new,
        "baseline_prob":   baseline,
        "final_p_true":    final_p_true,
        "kl_initial":      kl_initial,
        "kl_final":        kl_final,
        "delta_kl":        kl_final - kl_initial,
        "kl_history":      [round(k, 4) for k in kl_history],
        "p_new_history":   [round(p, 4) for p in p_new_history],
        "status":          "ok",
    }


# Snapshot original weights to CPU
print("Snapshotting original weights to CPU...")
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f"GPU free after snapshot: {free/1e9:.1f} GB — ready for ablation")


### 3.7 Step-Count Ablation: 1 / 5 / 10 / 20 with KL Active

Now with per-sample try/except — if one sample fails (OOM, NaN, whatever),
it gets logged and the run continues.

In [ ]:
STEP_COUNTS      = [1, 5, 10, 20]
LR               = 5e-3
BETA_KL          = 0.1
GRAD_CLIP        = 1.0
ablation_results = {s: [] for s in STEP_COUNTS}

print(f"Ablation config: lr={LR}, beta_kl={BETA_KL}, grad_clip={GRAD_CLIP}")
print(f"Neutral set size: {len(NEUTRAL_PROMPTS)}")
print(f"Step counts: {STEP_COUNTS}\n")

for n_steps in STEP_COUNTS:
    print(f"\n=== {n_steps}-step ablation on {N_SAMPLES} samples (KL active) ===")
    n_failed = 0
    for i in range(N_SAMPLES):
        try:
            restore_weights(model, original_state)
            ref_cache = cache_reference_logits(model, NEUTRAL_PROMPTS) if BETA_KL > 0 else None

            res = run_edit_with_kl_polished(
                model, cf_samples[i],
                circuit_log[i]["attn_heads"],
                circuit_log[i]["mlp_layers"],
                n_steps=n_steps, lr=LR,
                beta_kl=BETA_KL, grad_clip=GRAD_CLIP,
                ref_cache=ref_cache,
            )
            res["idx"]     = i
            res["n_steps"] = n_steps
            ablation_results[n_steps].append(res)

            del ref_cache
            torch.cuda.empty_cache()

            if i % 10 == 0:
                free = torch.cuda.mem_get_info()[0] / 1e9
                print(f"  [{i+1:>2}/{N_SAMPLES}]  "
                      f"edit={res['edit_success']:.4f}  "
                      f"kl={res['kl_final']:.3f}  "
                      f"gpu_free={free:.1f}GB")
        except Exception as e:
            n_failed += 1
            print(f"  [{i+1:>2}/{N_SAMPLES}]  FAILED: {type(e).__name__}: {str(e)[:80]}")
            ablation_results[n_steps].append({
                "idx": i, "n_steps": n_steps, "status": "failed",
                "error": f"{type(e).__name__}: {str(e)[:200]}",
                "edit_success": 0.0, "baseline_prob": 0.0,
                "final_p_true": 0.0, "kl_initial": 0.0, "kl_final": 0.0,
                "delta_kl": 0.0, "kl_history": [], "p_new_history": [],
            })
            torch.cuda.empty_cache()

    ok_results = [r for r in ablation_results[n_steps] if r.get("status") == "ok"]
    if ok_results:
        avg_e = sum(r["edit_success"] for r in ok_results) / len(ok_results)
        avg_b = sum(r["baseline_prob"] for r in ok_results) / len(ok_results)
        avg_k = sum(r["kl_final"]     for r in ok_results) / len(ok_results)
        flipped = sum(1 for r in ok_results if r["edit_success"] > 0.5)
        print(f"  SUMMARY  baseline={avg_b:.4f}  edit={avg_e:.4f}  delta={avg_e-avg_b:+.4f}")
        print(f"           kl_final={avg_k:.3f}  flipped={flipped}/{len(ok_results)}  failed={n_failed}")

restore_weights(model, original_state)

with open(RESULTS_DIR / "week5_ablation_kl_v2.json", "w") as f:
    json.dump({str(k): v for k, v in ablation_results.items()}, f, indent=2)
print(f"\nSaved -> {RESULTS_DIR}/week5_ablation_kl_v2.json")


### 3.8 Figures

Two figures for the methodology section:
- `fig_kl_vs_steps.png`: KL trajectory across step counts (with variance band)
- `fig_edit_vs_steps.png`: Edit success rate vs step count with % flipped

In [ ]:
def agg(results, key):
    vals = [r[key] for r in results if r.get("status") == "ok"]
    if not vals:
        return 0.0, 0.0
    mu = sum(vals) / len(vals)
    var = sum((v - mu) ** 2 for v in vals) / max(1, len(vals) - 1)
    return mu, var ** 0.5

# Figure 1 — KL trajectory across step counts
fig, ax = plt.subplots(figsize=(7, 4.5))
for n_steps, rs in ablation_results.items():
    ok = [r for r in rs if r.get("status") == "ok"]
    if not ok:
        continue
    # Pad histories so they all have length n_steps
    histories = [r["kl_history"] for r in ok if len(r["kl_history"]) == n_steps]
    if not histories:
        continue
    hist_arr = np.array(histories)   # [n_ok, n_steps]
    mu = hist_arr.mean(axis=0)
    sd = hist_arr.std(axis=0)
    steps_x = list(range(1, n_steps + 1))
    ax.plot(steps_x, mu, marker="o", label=f"{n_steps}-step run")
    ax.fill_between(steps_x, mu - sd, mu + sd, alpha=0.18)
ax.set_xlabel("Step")
ax.set_ylabel("KL(p_ref || p_edit)")
ax.set_title("KL trajectory during edit (mean ± 1 sd across 50 samples)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig_kl_vs_steps.png", dpi=160)
plt.show()
print(f"Saved -> {FIGS_DIR}/fig_kl_vs_steps.png")

# Figure 2 — Edit success + % flipped vs step count
steps_list   = sorted(ablation_results.keys())
edit_mu      = []
edit_sd      = []
flip_rate    = []
kl_mu_list   = []
for s in steps_list:
    ok = [r for r in ablation_results[s] if r.get("status") == "ok"]
    if not ok:
        edit_mu.append(0); edit_sd.append(0); flip_rate.append(0); kl_mu_list.append(0)
        continue
    mu, sd = agg(ok, "edit_success")
    edit_mu.append(mu); edit_sd.append(sd)
    flip_rate.append(sum(1 for r in ok if r["edit_success"] > 0.5) / len(ok))
    kl_mu_list.append(sum(r["kl_final"] for r in ok) / len(ok))

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.errorbar(steps_list, edit_mu, yerr=edit_sd, marker="o", color="C0",
             label="edit_success (mean ± sd)")
ax1.plot(steps_list, flip_rate, marker="s", color="C2",
         label="% samples flipped (P(target_new) > 0.5)")
ax1.set_xlabel("n_steps")
ax1.set_ylabel("Edit success / flip rate")
ax1.set_ylim(-0.05, 1.08)
ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(steps_list, kl_mu_list, marker="^", color="C3", linestyle="--",
         label="KL_final (mean)")
ax2.set_ylabel("KL_final", color="C3")
ax2.tick_params(axis="y", labelcolor="C3")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=9)
ax1.set_title("Edit success vs step count (KL-stabilized, Qwen3-0.6B)")
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig_edit_vs_steps.png", dpi=160)
plt.show()
print(f"Saved -> {FIGS_DIR}/fig_edit_vs_steps.png")


### 3.9 Harness-Ready Output for Person B

Schema is a superset of v1 — Aneesh's harness reads v1 fields and ignores
new ones, so it drops in with zero changes. New fields:
- `kl_initial`, `delta_kl` — drift measurement
- `kl_history`, `p_new_history` — per-step trajectories for plots
- `status`, `error` — so failed samples are identifiable

In [ ]:
rows, summary = [], {}
for n_steps, results in ablation_results.items():
    key = f"OurMethod_{n_steps}step"
    for r in results:
        rows.append({
            "method":          key,
            "model":           MODEL_NAME,
            "idx":             r["idx"],
            "n_steps":         n_steps,
            "edit_success":    round(r["edit_success"],  4),
            "baseline_prob":   round(r["baseline_prob"], 4),
            "over_extinction": None,
            "kl_initial":      round(r["kl_initial"],    4),
            "kl_final":        round(r["kl_final"],      4),
            "delta_kl":        round(r["delta_kl"],      4),
            "kl_history":      r["kl_history"],
            "p_new_history":   r["p_new_history"],
            "kl_active":       BETA_KL > 0,
            "beta_kl":         BETA_KL,
            "grad_clip":       GRAD_CLIP,
            "status":          r.get("status", "ok"),
            "error":           r.get("error", None),
        })

    ok = [r for r in results if r.get("status") == "ok"]
    if ok:
        summary[key] = {
            "avg_edit_success": round(sum(r["edit_success"] for r in ok) / len(ok), 4),
            "avg_baseline":     round(sum(r["baseline_prob"] for r in ok) / len(ok), 4),
            "avg_delta":        round(sum(r["edit_success"] - r["baseline_prob"] for r in ok) / len(ok), 4),
            "avg_kl_final":     round(sum(r["kl_final"]     for r in ok) / len(ok), 4),
            "avg_delta_kl":     round(sum(r["delta_kl"]     for r in ok) / len(ok), 4),
            "pct_flipped":      round(sum(1 for r in ok if r["edit_success"] > 0.5) / len(ok), 4),
            "n_ok":             len(ok),
            "n_failed":         len(results) - len(ok),
        }

harness_output = {
    "model":             MODEL_NAME,
    "kl_active":         BETA_KL > 0,
    "beta_kl":           BETA_KL,
    "grad_clip":         GRAD_CLIP,
    "neutral_set_size":  len(NEUTRAL_PROMPTS),
    "notebook_version":  "v2_polished",
    "rows":              rows,
    "summary":           summary,
}
with open(RESULTS_DIR / "week5_harness_output_kl_v2.json", "w") as f:
    json.dump(harness_output, f, indent=2)

print(f"{'Method':<28}  {'edit':>6}  {'delta':>7}  {'kl_fin':>7}  {'flip%':>6}  {'ok/fail':>8}")
print("-" * 82)
for method, s in summary.items():
    print(f"{method:<28}  {s['avg_edit_success']:>6.4f}  "
          f"{s['avg_delta']:>+7.4f}  {s['avg_kl_final']:>7.3f}  "
          f"{100*s['pct_flipped']:>5.1f}%  {s['n_ok']}/{s['n_failed']}")
print(f"\nShare week5_harness_output_kl_v2.json with Person B for live OE pass.")


### 3.10 Save and Download

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary_meta = {
    "author":           "Amogh",
    "notebook":         "Notebook 3 v2 — KL-Stabilized Reconsolidation (Polished)",
    "model":            MODEL_NAME,
    "timestamp":        timestamp,
    "n_samples":        N_SAMPLES,
    "step_counts":      STEP_COUNTS,
    "lr":               LR,
    "beta_kl":          BETA_KL,
    "grad_clip":        GRAD_CLIP,
    "kl_active":        BETA_KL > 0,
    "neutral_set_size": len(NEUTRAL_PROMPTS),
    "notebook_version": "v2_polished",
    "fixes_applied":    [
        "kl_final measurement moved after loop (fixes 1-step kl=0 bug)",
        "Per-step KL and p_new trajectory tracking",
        "Gradient clipping at max_norm=1.0",
        "Per-sample try/except for robustness",
        "Figures saved inline (kl_vs_steps, edit_vs_steps)",
    ],
    "ablation_summary": summary,
}

with open(RESULTS_DIR / f"summary_nb3v2_{timestamp}.json", "w") as f:
    json.dump(summary_meta, f, indent=2)

# Include figures in the zip
all_results_dir = Path("zip_contents")
all_results_dir.mkdir(exist_ok=True)
for p in RESULTS_DIR.glob("*"): shutil.copy2(p, all_results_dir / p.name)
for p in FIGS_DIR.glob("*"):    shutil.copy2(p, all_results_dir / p.name)

zip_path = f"/content/PersonA_Notebook3v2_results_{timestamp}"
shutil.make_archive(zip_path, "zip", all_results_dir)

print("=" * 70)
print(f"  NOTEBOOK 3 v2 RESULTS — Amogh  ({timestamp})")
print("=" * 70)
print(f"  KL active     : {BETA_KL > 0}  (beta={BETA_KL}, grad_clip={GRAD_CLIP})")
print(f"  Step counts   : {STEP_COUNTS}")
print(f"  Samples       : {N_SAMPLES}")
print()
for method, s in summary.items():
    print(f"  {method:<24}  edit={s['avg_edit_success']:.4f}  "
          f"flip%={100*s['pct_flipped']:>5.1f}  kl={s['avg_kl_final']:.2f}  "
          f"n_ok={s['n_ok']}")
print()
print("  Files in zip:")
for f in sorted(all_results_dir.glob("*")):
    print(f"    {f.name:<40}  {f.stat().st_size//1024:>4} KB")
print(f"\n  Download: {zip_path}.zip")
print("=" * 70)

from google.colab import files
files.download(f"{zip_path}.zip")
print("\nDONE. Upload week5_harness_output_kl_v2.json to Aneesh's Notebook B")
print("for live OE measurement (DRY_RUN=False on A100).")
